In [1]:
import re
import json
import pandas as pd
import numpy as np
from datetime import datetime
import pycountry
from IPython.display import display, Markdown
from numpyro import distributions as dist
import yaml as yml
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
import git
import yaml as yml

from emu_renewal.constants import BASE_PATH, OUTPUTS_PATH
from emu_renewal.selection import get_mob_avail_countries, gather_who_data, find_absent_inds, \
    find_neg_inds, find_outliers, find_nans_repeats
from emu_renewal.inputs import get_worldbank_national_pop, get_income_group, get_fb_visited_mobility, \
    get_fb_singletile_mobility, process_raw_google_mobility, get_google_mobility, get_country_pop, \
    get_undesa_national_pop, get_country_vacc_data, get_world_shp
from emu_renewal.outputs import get_table_df_from_priors_dict, add_bool_row_to_table, add_mob_avail_to_world
from emu_renewal.document import get_func_blurb
from emu_renewal.indicators import get_deaths_target, get_cases_target, filter_seroprev, get_owid_hosps, \
    get_all_seroprev, get_seroprev_pooled_totals, get_var_target, extract_specific_var, get_country_vars, \
    get_alpha_info, get_delta_info, get_specific_var_props, \
    get_ba2_info, get_ba5_info
from emu_renewal.run import find_run_start_time, find_run_end_time, get_hosp_target, get_seroprev_target, \
    get_mobility_provider, run_calibration
from emu_renewal.geospatial import FacebookMobilityBuilder
from emu_renewal.renew import MultiStrainModel
from emu_renewal.distributions import GammaDens
from emu_renewal.process import _get_cos_curve_at_x
from emu_renewal.mobility import NoMobilityProvider, WeightedExpMobilityProvider, SingleSeriesExpMobilityProvider
from emu_renewal.calibration import custom_init, StandardCalib
from emu_renewal.priors import get_standard_priors
from emu_renewal.plotting import plot_beta_priors, plot_duration_params, plot_inclusion

plt.style.use("default")
set_matplotlib_formats("svg")

{{< pagebreak >}}

# Documentation

In [2]:
repo = git.Repo(search_parent_directories=True)
commit_id = repo.git.rev_parse("--short", "HEAD")
txt = (
    "The following document was documented algorithmically "
    "from the code used in the analysis. "
)
txt += f"This version was produced from commit {commit_id}. \n\n"
txt += (
    "The rationale for our methods is provided in the "
    "Methods section of the main manuscript.\n\n"
)
txt += "# Analysis methods \n\n"
display(Markdown(txt))

The following document was documented algorithmically from the code used in the analysis. This version was produced from commit bf2b58b. 

The rationale for our methods is provided in the Methods section of the main manuscript.

# Analysis methods 



## Summary equation
The mechanics of the renewal process can be summarised in the following equations, the detail of which is expanded in the following sections.
$$i_{p,v,t}=n_{p,t}x_{p,v}(1-e^{-\lambda_{v,t}})$$
$$\lambda_{v,t}=\beta\sigma_{v}e^{W_{t}}M_{t}\sum_{\tau=t-50}^{t-1}{(s_{v,\tau}+\sum_{p'}i_{p',v,\tau})g_{t-\tau}}$$
$$n_{p,t}=n_{p,t-1}-\sum_{v}i_{p,v,t}+i_{\tilde{p},\tilde{v},t}-qzn_{p,t-1}+qzr_{p}\sum_{p'}{n_{p',t-1}}$$
$$M_{t}=
\begin{cases}
1 & \text{if no mobility} \\
(\frac{\sum_{l}w_{l}G_{l,t}}{\sum_{l}w_{l}})^{m} & \text{if Google mobility} \\
F_{t}^{m} & \text{if Facebook mobility} \\
\end{cases}
$$
$$o_{t}=a\sum_{v}(\prod_{v'=v_{1}}^{v}{y_{v'}})\sum_{p}\sum_{\tau=1}^{50}i_{p,v,t-\tau} \times c_{\tau}$$
$$h_{t}=\sum_{\tau=1}^{50}o_{t-\tau}(1-\sum_{u=0}^{\tau-1}d_{u})$$
Where:

- $i_{p,v,t}$ represents the incidence of variant $v$ in immunity status group $p$ at time $t$, where $p$ represents the immunity status prior to infection with variant $v$
- $v$ takes sequential values from $v_1$ to $v_V$, where $V$ is the number of variants included in the analysis
- $p\in \{0, 1\}^{V}$ are the possible combination of past variant exposure statuses with $V$ variants
- $n_{p,t}$ is the size of the population with immunity status $p$ at time $t$
- $q$ indicates whether waning immunity is incorporated (i.e. $q=0$ base case analysis, $q=1$ waning immunity sensitivity analysis)
- $z$ is the reciprocal of the duration of natural immunity
- $r_{p}$ indicates whether $p=\{0\}^V$ and represents the infection-naive population (i.e. $r=1$ if $p=\{0\}^V$ and $r=0$ otherwise)
- $x_{p,v}$ is the relative susceptibility of persons with immunity status $p$ to infection with variant $v$
- $x_{\{0\}^{V},v} = 1$, full susceptibility for those never previously infected
- $x_{p,v} = 0$ for infection with a variant to which the person has previously been infected with or a preceding variant (e.g. infection with Alpha following infection with Delta)
- $x_{p,v}$ for immunity statuses other than the two preceding cases is the complement of a single calibrated cross-immunity parameter with a beta-distributed prior
- $\beta$ is the infectiousness of the starting variant
- $\sigma_{v}$ is the infectiousness of variant $v$ relative to the starting variant
- $W_{t}$ is the non-mechanistic residual transmission process, under which a Wiener sequence of updates from a starting value of zero is linked with a cosine spline at time $t$
- $s_{v,t}$ is the rate of seeding with variant $v$ at time $t$
- $g_{t}$ is $\int_{t}^{t+1} g(u)\,du$, where $g$ is the probability density function of the generation interval distribution
- $\tilde{v}$ is the variant for which infection results in transition from immunity status $\tilde{p}$ to immunity status $p$
- $G_{l,t}$ is the Google mobility estimate at location $l$ and time $t$
- $F_{t}$ is the Facebook mobility estimate (either tiles visited or single tile) at time $t$
- $w_{l}$ is a calibrated location-specific weight parameter with support $[0, 1]$
- $o_{t}$ is any one of the incident indicator quantities calculated from the incidence time series, i.e. either deaths, case notifications, hospital admission or ICU admissions
- $y_{v}$ is the relative severity of the infecting variant relative to the preceding variant, with $y_{1}=1$ (i.e. the severity of the starting strain is set to one)
- $c_{t}$ is $\int_{t}^{t+1} c(u)\,du$, where $c$ represents the probability density function of the distribution of the delay from the onset of the infection epiosde to the onset of the incident indicator considered
- $a$ is the proportion of infection episodes resulting in the indicator considered
- $h_{t}$ is either one of the prevalent indicators calculated from the incident indicators, i.e. either hospital occupancy or ICU occupancy
- $d_{t}$ is $\int_{t}^{t+1} d(u)\,du$, where $d$ represents the probability density function of the distribution of the time from the onset of the incident indicator (either hospital or ICU admission) to the end of the prevalent indicator period (either hospital or ICU discharge)

In [3]:
txt = "\n\n## Renewal model\n"
model_rationale = (
    "We developed a discrete renewal model "
    "with daily innovations. The core aspect of our "
    "approach was the convolution of the preceding "
    "strain-specific incidence rate with "
    "the generation interval for subsequent infections, "
    "adjusted for past immunity. "
    "We then applied convolutions to this model "
    "to estimate model outputs that could be compared "
    "against our indicator targets described below.\n\n"
)
txt += model_rationale
dummy_start = datetime(2020, 1, 1)
dummy_end = datetime(2021, 12, 31)
no_mob_provider = NoMobilityProvider()
dummy_model = MultiStrainModel(1.0, dummy_start, dummy_end, [""], dummy_start, no_mob_provider, False, False)
txt += get_func_blurb(dummy_model.renew)

txt += "\n\n### Outputs\n"
txt += get_func_blurb(dummy_model.renewal_func) + "\n\n"
dummy_dist = GammaDens()
txt += get_func_blurb(dummy_dist.get_params) + "\n\n"

txt += "## Modelled population size \n\n"
txt += get_func_blurb(get_country_pop)
txt += get_func_blurb(get_worldbank_national_pop)
txt += get_func_blurb(get_undesa_national_pop)
txt += "\n\n## Analysis period\n"
analysis_period_rationale = (
    "For all countries, we aimed to place our analysis period "
    "during a time period for which the variation in transmission rate "
    "may have been substantially attributable to changes in mobility, "
    "although we developed an approach (described below) to address "
    "the consideration that the emergence of new SARS-CoV-2 variants "
    "likely contributed substantially. "
    "Therefore, we aimed to select time periods during which "
    "the roll-out of vaccination programs would have contributed "
    "less to variations in transmission rate.\n\n"
    "For all countries other than Oceania and Singapore, we addressed this "
    "by limiting our analysis period to the time prior to "
    "vaccination reaching levels that may have had a significant "
    "population-level effect.\n\n"
    "For most countries, a single base case no mobility analysis was run, "
    "against which the Google and Facebook analyses could be compared. "
    "However, for Oceania and Singapore, the analysis period for the Facebook "
    "analysis was shorter due to the discontinuation of Facebook "
    "data after May 2022. For these countries, an additional "
    "no mobility comparison analysis was run. "
)
txt += analysis_period_rationale
txt += get_func_blurb(find_run_start_time) + "\n\n"
txt += get_func_blurb(find_run_end_time)
txt += get_func_blurb(get_country_vacc_data)
txt += "\n\n## Mobility\n"
mobility_rationale = (
    "The main purpose of our analysis was to understand "
    "the effect of empirically observed changes in mobility "
    "on the observed fluctuations of the COVID-19 epidemic. "
    "We addressed this by including various mobility observation "
    "streams available from Big Tech companies within our model.\n\n"
)
txt += mobility_rationale
txt += get_func_blurb(get_mobility_provider)
txt += "\n\n### No mobility\n"
txt += get_func_blurb(NoMobilityProvider.__init__)
txt += "\n\n### Google mobility\n"
n_domains = 6
dummy_g_mob = pd.DataFrame(1.0, index=range(10), columns=range(n_domains))
dummy_g_priors = {"mob_weights": dist.Uniform(np.zeros(n_domains), np.ones(n_domains)), "mob_exp": None}
weighted_exp_mob_provider = WeightedExpMobilityProvider(dummy_g_mob, dummy_g_priors)
txt += get_func_blurb(process_raw_google_mobility)
txt += get_func_blurb(get_google_mobility)
txt += get_func_blurb(weighted_exp_mob_provider.__init__)
txt += get_func_blurb(weighted_exp_mob_provider.get_parameterised_mobility)
txt += "\n\n### Facebook mobility\n"
dummy_fb_mob_builder = FacebookMobilityBuilder()
txt += get_func_blurb(dummy_fb_mob_builder.build_mobility) + "\n\n"
txt += get_func_blurb(get_fb_visited_mobility)
txt += get_func_blurb(get_fb_singletile_mobility) + "\n\n"
dummy_fb_mob = pd.Series(1.0, index=range(10))
dummy_fb_priors = {"mob_exp": None}
single_series_mob_provider = SingleSeriesExpMobilityProvider(dummy_fb_mob, dummy_fb_priors)
txt += get_func_blurb(single_series_mob_provider.get_parameterised_mobility)
txt += get_func_blurb(single_series_mob_provider.__init__)
txt += "\n\n## Residual transmission process\n"
process_rationale = (
    "We included a non-mechanistic residual process "
    "within our model, which was intended to capture "
    "any variations in transmission that could not be "
    "attributed to the explicitly modelled processes "
    "(variant strain emergence, changes in mobility "
    "and the development of population immunity).\n\n"
)
txt += get_func_blurb(dummy_model.fit_process_curve)
txt += get_func_blurb(_get_cos_curve_at_x)
txt += get_func_blurb(dummy_model.initialise_var_proc)
txt += "\n\n## Calibration targets\n"
indicator_rationale = (
    "We only included countries for which multiple "
    "data streams were available to estimate variations "
    "in transmission rates over time. "
    "Specifically, we required that a time series for both "
    "COVID-19-attributable deaths and case notifications were available. "
    "However, we also included one time series for a hospital-related "
    "indicator, where this could be sourced. "
    "Because we considered that the emergence of new "
    "SARS-CoV-2 variants could have substantially affected transmission, "
    "we included explicit calibration targets for the "
    "strain-specific proportional prevalence of variants where available. "
    "Although possibly less epidemiologically relevant to "
    "this analysis, we sought to broadly capture the "
    "overall size of the epidemic. This was addressed by "
    "explicitly incorporating susceptible depletion and "
    "including seroprevalence estimates as calibration targets where available. "
    "We did not include vaccination explicitly within the model "
    "because our approach to country and time period inclusion "
    "was intended to focus on time periods during which "
    "vaccination would not have substantially influenced transmission rates.\n\n"
)
txt += indicator_rationale
txt += "\n\n### WHO indicators\n"
txt += get_func_blurb(get_deaths_target)
txt += get_func_blurb(get_cases_target)
txt += "\n\n### Hospitalisations\n"
txt += get_func_blurb(get_owid_hosps)
txt += get_func_blurb(get_hosp_target) + "\n\n"
txt += "\n\n### Seroprevalence\n"
txt += get_func_blurb(get_all_seroprev)
txt += get_func_blurb(filter_seroprev)
txt += get_func_blurb(get_seroprev_pooled_totals) + "\n\n"
txt += get_func_blurb(get_seroprev_target) + "\n\n"
txt += get_func_blurb(get_income_group)
txt += "\n\n### Variants\n\n"
variant_rationale = (
    "We included up to three key variants explicitly within "
    "our analysis framework. For all countries "
    "other than Oceania and Singapore "
    "our approach was intended to represent strains prior to the "
    "emergence of the Alpha variant and strains of Alpha "
    "with strains of Delta also included where sufficient data "
    "were available and the time period simulated meant that "
    "the emergence of Delta was relevant to the simulation. "
    "For Oceania and Singapore, we included pre-BA.2 strains of Omicron, "
    "along with Omicron BA.2 and Omicron BA.5. \n\n"
)
txt += "#### Data extraction\n"
txt += get_func_blurb(get_country_vars)
txt += get_func_blurb(extract_specific_var)
txt += get_func_blurb(get_specific_var_props) + "\n\n"
txt += get_func_blurb(get_var_target)
txt += "\n\n#### Alpha\n"
txt += get_func_blurb(get_alpha_info)
txt += "\n\n#### Delta\n"
txt += get_func_blurb(get_delta_info)
txt += "\n\n#### Omicron BA.2\n"
txt += get_func_blurb(get_ba2_info)
txt += "\n\n#### Omicron BA.5\n"
txt += get_func_blurb(get_ba5_info)

In [4]:
display(Markdown(txt))



## Renewal model
We developed a discrete renewal model with daily innovations. The core aspect of our approach was the convolution of the preceding strain-specific incidence rate with the generation interval for subsequent infections, adjusted for past immunity. We then applied convolutions to this model to estimate model outputs that could be compared against our indicator targets described below.

 For all analyses, the starting population was assigned to the fully susceptible category. 

### Generation interval

 A gamma-distributed generation interval was used for the renewal process. This is consistent with an investigation of household clusters from Germany that showed the profile of serial intervals was well matched by a gamma distribution.[@anderheiden2022] The generation interval was truncated at 50 days, because the density reached negligible values beyond this point. 

### Renewal process

 The strain-specific incidence array updated for strain seeding was convolved with the generation interval distribution vector to create an array of the effective number of infectious individuals for each strain. These quantities were then multiplied by scalar values representing residual transmission scaling and adjustment for mobility and divided through by the population size (to obtain the scaled per capita infectious population). This was multiplied by the strain-specific vector for the relative infectiousness of each strain to calculate a per capita rate of infection. We then determined the actual force of infection for infection-naive persons for each strain as $1 - e^{-\lambda}$ (where $\lambda$ represents the calculated infection rate) to ensure that the per capita risk of infection could not exceed one in a time step. 

### Variant seeding

 Each strain (including the starting strain) was seeded into the model using a triangular pulse of new infections for which the peak per capita seeding rate was specified as a parameter. At each calculation day, the new strain-specific seeding values were added to the most recent value for the strain-specific history of incidence. Infectiousness of each variant was specified with reference to the first modelled variant strain. 

### Immunity

 The strain-specific force of infection was then applied to each possible immunological past history of preceding exposure to variant combinations and their associated susceptibility to infection to calculate the number of people infected from each category and transition them to their new immunological states. Persons who had never previously been infected with any strain were considered to have no immunological protection, and the rate of infection was not adjusted further. We considered partial cross immunity was provided by infection with a preceding variant strain to infection with subsequent strains. However, previously infected persons were assumed to derive complete immunity to the infecting strain and to variant strains that had emerged prior to the infecting strain (for example, past infection with Delta conferred complete immunity against future infection with Alpha). 

### Waning immunity

 For our sensitivity analyses in which waning host-related immunity from past infection was incorporated, individuals were transitioned from each immune category into the fully susceptible population at a constant rate represented by the reciprocal of the duration immune. 

### Outputs
 Once an iteration of the renewal analysis had been run to calculate incidence values over time, the other epidemiological outputs were calculated using a series of convolution operations applied to this series. Cases were calculated by convolving the total incidence summed over strains with a gamma-distributed set of delays from incidence to notification which was truncated after 50 days. This was then multiplied through by the case detection rate (a proportion) and weekly cases were calculated by summing over the preceding 7 days.

 By contrast, deaths were calculated from the strain-specific incidence to allow for strain-specific severity for analyses considering the pre-Omicron/pre-vaccination period. The severity of each variant was interpreted with reference to the preceding modelled variant, such that the severity of each variant relative to the starting variant was calculated as the cumulative product of the severity parameter for each modelled emerging variant. Strain-specific variant severity was not included for the Omicron era analyses that considered Omicron subvariants BA.1, BA.2 and BA.5. A separate set of parameters governed the time from incidence to death with the fraction of incident episodes resulting in death given by the infection fatality rate adjusted for strain-specific severity. For Singapore and countries of Oceania, a reduction in the risk of death was applied because this analysis was performed after wide-scale population vaccination had been achieved and Omicron sub-variants were the dominant strains, with severity assumed to be equivalent for all Omicron sub-variants. The relative reduction in the risk of death during the later Omicron/vaccination period was set according to a parameter specifying the protection of vaccination against death given infection and was not varied during calibration, because this would have been collinear with the risk of death parameter.

 As for cases and deaths, hospitalisations were estimated through a convolution distribution with independently calibrated parameters, a hospital admission fraction, but the same strain-specific severity parameters as for deaths. As for the approach to deaths, strain-specific severity was included for the pre-Omicron period analyses, and a reduction in the risk of hospitalisation was applied to account for vaccination for the Omicron period analyses. The relative reduction in the risk of hospitalisation was set according to a vaccination protection against hospitalisation given infection parameter. As for cases, deaths and hospitalisations, the convolution of time to ICU admission was parameterised independently.

 Hospital and ICU occupancy were obtained by convolving the time series of hospital and ICU admissions with the complement of the cumulative distribution of the time to hospital or ICU discharge. For comparison to serosurveillance data, seropositivity was calculated as the complement of as the proportion of the population remaining in the entirely infection-na&iuml;ve immunity sub-population. Finally, the proportion of incidence attributable to each variant strain was calculated from the strain-specific incidence. 

 The parameters to each gamma distribution used in our anlaysis were parameterised by analytically calculating the "a" (shape) and scale parameters from the mean and standard deviation determined by our literature review. 

## Modelled population size 

 We used the total population size estimated by the World Bank where a population size was available from this source, and used the estimate provided by the United Nations Department of Economic and Social Affairs otherwise.  Population data was downloaded from [the World Bank](https://databank.worldbank.org/source/population-estimates-and-projections#) on 1 April 2025. The population size for 2020 was used for all countries except for Singapore and countries of Oceania, for which the population size in 2022 was used (because of the later analysis period for these countries).  [UN DESA population data](https://population.un.org/wpp/assets/Excel%20Files/1_Indicator%20(Standard)/EXCEL_FILES/2_Population/WPP2024_POP_F02_1_POPULATION_5-YEAR_AGE_GROUPS_BOTH_SEXES.xlsx) was downloaded on 18 March 2025 and where UN DESA population was needed, data for 2020 was used. 

## Analysis period
For all countries, we aimed to place our analysis period during a time period for which the variation in transmission rate may have been substantially attributable to changes in mobility, although we developed an approach (described below) to address the consideration that the emergence of new SARS-CoV-2 variants likely contributed substantially. Therefore, we aimed to select time periods during which the roll-out of vaccination programs would have contributed less to variations in transmission rate.

For all countries other than Oceania and Singapore, we addressed this by limiting our analysis period to the time prior to vaccination reaching levels that may have had a significant population-level effect.

For most countries, a single base case no mobility analysis was run, against which the Google and Facebook analyses could be compared. However, for Oceania and Singapore, the analysis period for the Facebook analysis was shorter due to the discontinuation of Facebook data after May 2022. For these countries, an additional no mobility comparison analysis was run.  For these countries, the start of the calibration period was set to be the time at which the per capita daily rate of deaths passed 2 deaths per million population. However, if this threshold was not reached by 1 June 2020, the simulation commenced at this default time instead. For Singapore and countries of Oceania, the simulation commenced from the time that vaccination coverage reached 90% of its final value. 

 For all countries other than Singapore and those of Oceania, the end time for the analysis was calculated as the time that population vaccination coverage passed 5%, provided that vaccination coverage did reach this value before the default end time of 1 December 2021. Otherwise, this default end date was used instead. For Oceania and Singapore, the latest date for which Google mobility data was available was used.  We substituted Germany's data for Switzerland and Ireland because these two countries had almost identical profiles of vaccine doses administered per person in the earliest phases of the roll-out and vaccination data was not available for these two countries through the period when coverage approached the analysis start threshold. Similarly, we substituted Great Britain for Qatar based on the same rationale. 

## Mobility
The main purpose of our analysis was to understand the effect of empirically observed changes in mobility on the observed fluctuations of the COVID-19 epidemic. We addressed this by including various mobility observation streams available from Big Tech companies within our model.

 For each country, we ran one analysis with no mobility scaling to the transmission rate. We further ran one analysis in which Google mobility was used to scale the transmission rate, if mobility data was available from Google. For countries for which any of the Google mobility domains reached an average value of 2 or above during the last 30 (i.e. 2 times the starting value which is normalised to one), we additionally ran a detrended Google mobility analysis. Under this detrended analysis, each Google location was individually detrended by dividing through by a linear estimate of the increase in that mobility domain over time. We also ran two analyses in which Facebook mobility was used to scale the transmission rate, if mobility data was available from Facebook. Although Apple mobility data was available and we were able to run analyses using this data source, Apple's terms of use indicate that this source of data cannot be used for this purpose. We contacted Apple, who declined to allow their data to be used for this project. For all mobility sources, we smoothed the raw data using a 7-day centred rolling average. For all analyses incorporating mobility scaling, we used an exponential scaling parameter (described in more detail below) which was assigned a uniform prior over domain [0, 2]. 

### No mobility
 For the analysis without mobility, no empiric data was used to scale the transmission rate over time (which was implemented by setting the mobility scaling value to one throughout the simulation). 

### Google mobility
 We obtained mobility data from [the Google Community Mobility Reports](https://www.gstatic.com/covid19/mobility/Region_Mobility_Report_CSVs.zip) on 14 January 2025 and extracted national mobility estimates by Google mobility domain.  For this analysis, we used the raw value interpreted as a percentage plus one to scale the transmission rate.  This mobility analysis type used a set of priors weighting each mobility domain, along with one further prior governing the overall effect of the weighted mobility estimate in scaling the transmission rate.  The weights for each mobility domain were normalised to sum to one, with the resulting weighted mobility profile exponentiated to the value specified by the mobility exponential parameter. 

### Facebook mobility
 For each geographic region included in the Facebook data, we estimated a population by intersecting polygons with the centroid of population data grids, and then weighted the resulting series by these calculated populations. For the small proportion of (low-population) time series that had missing data, we imputed population estimates by nearest neighbour interpolation. In general, these series were found to have a negligible contribution to the final outputs. 

 We used the `all_day_bing_tiles_visited_relative_change` for the first Facebook mobility analysis, and scaled transmission transmission according to one plus this mobility metric.  For the second Facebook mobility analysis, we used the `all_day_ratio_single_tile_users` estimate and scaled transmission according to one minus this mobility metric. 

 Because weighting was note required, only one mobility-related parameter was needed under each of these approaches, which specifies the mobility exponent value for the effect of the mobility data in scaling the transmission rate.  This mobility approach was used for each of the two analyses that incorporated Facebook data, using both the tiles visited and the within tile estimates. 

## Residual transmission process
 A Wiener variable process was used to capture variation in transmission over time. The starting value for this process was fixed at zero and the subsequent updates to the process were explored in calibration. This exploration was performed in logarithmic space, with the calibrated values for each update exponentiated before being used to scale the transmission rate. Each parameter pertaining to the updates to residual transmission scaling was assigned the same prior centred at zero (i.e. no change), and so can be interpreted as the change in the log-transformed residual transmission scaling relative to the previous value.  The cosine function was obtained by translating and scaling a half cosine function (i.e. a cosine function with support $[0, \pi]$), such that it intersected the starting point $(t_{1}, y_{1})$ and finishing point $(t_{2}, y_{2})$ with a gradient of zero at both of these points. This choice of fitting approach ensured that the residual transmission scaling function, its derivative and its higher order derivatives are continuous.  Each time point for fitting the residual transmission scaling process was set at intervals through the analysis period spaced by 7 days working backwards from the end of the analysis period. The scaling process was then fit to these points using piecewise cosine functions from a starting value of zero. 

## Calibration targets
We only included countries for which multiple data streams were available to estimate variations in transmission rates over time. Specifically, we required that a time series for both COVID-19-attributable deaths and case notifications were available. However, we also included one time series for a hospital-related indicator, where this could be sourced. Because we considered that the emergence of new SARS-CoV-2 variants could have substantially affected transmission, we included explicit calibration targets for the strain-specific proportional prevalence of variants where available. Although possibly less epidemiologically relevant to this analysis, we sought to broadly capture the overall size of the epidemic. This was addressed by explicitly incorporating susceptible depletion and including seroprevalence estimates as calibration targets where available. We did not include vaccination explicitly within the model because our approach to country and time period inclusion was intended to focus on time periods during which vaccination would not have substantially influenced transmission rates.



### WHO indicators
 The number of deaths by week reported by WHO was used as a calibration target for all included countries. Any values of zero in this series were replaced with a value of 0.5 to enable comparison of the deaths target to our modelled outputs in logarithmic space. The log of the reported target deaths value was compared against modelled deaths by calculating the density of a normal distribution centred at the log-transformed modelled value. Deaths was one of up to three time-series indicators for which a common dispersion parameter was used for this normal distribution for the target comparison. The value for weighting the deaths indicator time series was set to 20 (a quantity that can be interpreted in relation to the weights of the other indicators, but whose absolute size is arbitrary).

  The number of cases by week reported by WHO was included as the second calibration target for all countries. As for deaths, any zero values were replaced with a value of 0.5. We only calibrated to cases from 1 June 2020 onward, because we considered that prior to this time many countries were still rapidly scaling up testing capacity and this indicator may have been less reliable. Cases was the second indicator for which a common dispersion parameter was applied. A total weight was calculated for the overall time-series of cases such that the weight for each individual observation (date) of the cases time-series was equal to that for each death observation. 

### Hospitalisations
 A maximum of one hospitalisation-related indicator for calibration was chosen using a hierarchical approach. In selecting the indicator, the number of new admissions was preferred over estimates of total bed occupancy, and total hospital indicators were preferred over ICU indicators. As such, the final hierarchy of indicators was: 

1. New hospital admissions 

2. Hospital occupancy 

3. New ICU admissions 

4. ICU occupancy 

5. No hospital indicator 



 That is, the highest ranked indicator was used based on data availability, and no hospital indicator was incorporated if none was available.  This indicator was the final calibration target for which a common dispersion parameter was applied. As for the cases target, a weight was applied to the overall hospitalisation time series such that the weight for each observation point was equal to that for each death observation. 



### Seroprevalence
 Seroprevalence data was obtained from [SeroTracker](https://github.com/serotracker/sars-cov-2-data/raw/refs/heads/main/serotracker_dataset.csv) on 11 December 2024, with the date for each serosurvey estimate calculated as the mid-point between the reported start and end sampling dates for each survey. This date was then lagged earlier by 14 days for comparison to modelled outputs to account for the delay between infection and the subsequent development of detectable antibodies.  We filtered the SeroTracker data to include only the estimate reported as primary from Unity-aligned national-level surveys for which the number of participants was at least 600. We also considered only a maximum of one seroprevalence value for any given date (selecting only the largest of three surveys done on the same day for Mexico).  For any consecutive estimates for which a lower estimate followed an immediately preceding higher estimate, we pooled these two estimates and placed the pooled estimate at the mid-point of the dates of the two estimates. We recursively applied this process until seroprevalence estimates were monotonically increasing over time. 

 We compared the modelled proportion ever infected against the seroprevalence reported at least six months (183 days) after the start of the simulation. We chose to exclude seroprevalence estimates from the early months of the pandemic period because the seropositive proportions reported at this time were less likely to be comparable to our model outputs. This is because our modelled seroprevalence would remain close to zero with plausible parameter values for some time after the start of the analysis, whereas seroprevalence estimates could reach higher values due to factors that include low levels of transmission prior to the analysis period and sampling bias. In contrast, we considered that later seroprevalence should provide a broad indication of epidemic size. For countries for which seroprevalence calibration targets were used, we assigned a target weight to this indicator of 5 (which is an arbitrary quantity in absolute terms, but can be interpreted with reference to the deaths indicator weight of 20). The seroprevalence target had a dispersion parameter that was normally distributed and independent of the dispersion parameters for the other targets, including the shared dispersion parameter of the time series indicators. 

 We discarded seroprevalence estimates that fell less than 5% away from a value of zero or 100%. We also ignored seroprevalence estimates for Oceania and Singapore, for which the analysis was run largely through 2022, during which time seroprevalence values would not reflect the attack rate during the simulation period. We further ignored seroprevalence estimates from low and lower middle income countries of Africa, because we were unable to obtain good fits for several of these countries while also maintaining plausible detection/severity parameters (i.e. case detection rate, hospital admission rate and infection fatality rate). That is, we applied much lower values for these detection-related parameters in these countries, although the modelled attack rate still often remained below seroprevalence estimates for some countries. 

 Country income classifications were obtained from [the World Bank](https://datacatalogapi.worldbank.org/ddhxext/ResourceDownload?resource_unique_id=DR0090755). 

### Variants

#### Data extraction
 Reports of the number of isolates of specific variants of SARS-CoV-2 were obtained from the [covariants](https://github.com/hodcroftlab/covariants/raw/refs/heads/master/cluster_tables/) GitHub repository. Each variant-specific file was downloaded and used to create country-specific tables of the variant-specific counts by date.  The terms used to identify variants prior to Alpha were: 20A.EU1, 20A.EU2, 20B.S.732A, 21C.Epsilon. The text "Delta" was used to identify Delta variants and the text "21L.Omicron" was used to distinguish the BA.2 variant from later variants (modelled as BA.5).  Variant data was considered for dates on which at least 5 sequences were available for the country considered. Further, we required at least 5 such dates be available for that country's variant data to be used as a calibration target. 

 To obtain data to use as calibration targets for the strain-specific proportional prevalence of both the Alpha and the Delta variants, we used national data for the country analysed where available. If data was not available for the country considered or if all values for the proportion of the variant were one, we used pooled data from all the other countries from the same continent where available. If insufficient data on a particular variant was available for the country or its continent, prevalence of that variant was not included as a calibration target. 

#### Alpha
 For countries other than Singapore and countries of Oceania and Africa, a target for the Alpha variant was included in our calibration algorithm. Calibration against data for Alpha was permitted from the beginning of the simulation period (from 1 January 2020) for these countries. The periods for calibration against the Alpha and the Delta variants were set so as to be mutually exclusive in time. Specifically, the date for transitioning from calibrating against available data for the Alpha to calibrating against data for Delta was set as 1 March 2021. Exceptions were made for several Asian countries for which this transition date was set one month earlier and two countries of North America for which it was set six weeks later. (Specifically, the exceptions were Afghanistan: 1 February 2021, India: 1 February 2021, Nepal: 1 December 2020, Indonesia: 1 February 2021, Saudi Arabia: 1 February 2021, Oman: 1 February 2021, Kuwait: 1 February 2021, South Korea: 1 February 2021, Malaysia: 1 February 2021, Honduras: 15 April 2021, Puerto Rico: 15 April 2021.) If this date occurred after the end of the simulation, the Alpha calibration period continued to the end of the simulation. Comparison of the log of the target estimate against the log of the modelled value was undertaken using a normal distribution with a single dispersion parameter that applied to all modelled variants (but was independent of the time series dispersion parameter). The target weight for the Alpha target was set to 5. 

#### Delta
 For countries other than those of Oceania and Singapore, the Delta variant was included if the end date of the calibration fell later than 1 May 2021. The target weight for calibration to Delta was set to 5 for most countries. Exceptions were made if the target time series for Delta began towards the very end of the calibration (last 25 days), in which case a higher weight (of 25) was needed to achieve a plausible fit to the profile of the emergence of this variant with fewer data points. 

#### Omicron BA.2
 A calibration target for Omicron BA.2 was only included for Oceania and Singapore. As for other variants, the target weight for BA.2 was set to 5. The BA.2 calibration target for these countries used the data available from 1 January 2022 to 15 April 2022. 

#### Omicron BA.5
 As for BA.2, a calibration target for Omicron BA.5 was only included for Oceania and Singapore. As for other variants, the target weight for BA.5 was set to 5. The BA.5 calibration target for these countries used the data available from 1 April 2022 to 1 September 2022. 

In [5]:
txt = "\n\n## Calibration algorithm\n"
calibration_rationale = (
    "We used a gradient-based algorithm "
    "for model calibration, which efficiently "
    "explored our multidimensional parameter space. "
    "By exploring all time series in log-space "
    "with a common dispersion parameter, "
    "we were able to apply an algorithm that "
    "provided epidemiologically plausible results for all "
    "countries simulated without the need for "
    "extensive country-specific adaptations.\n\n"
)
txt += calibration_rationale
txt += "\n\n### Initialisation\n"
txt += get_func_blurb(custom_init)
txt += "\n\n### Parameter processing\n"
dummy_calib = StandardCalib(dummy_model, {}, {})
txt += get_func_blurb(dummy_calib.sample_calib_params)
txt += get_func_blurb(dummy_calib.__init__)
txt += "\n\n"
txt += get_func_blurb(get_standard_priors)
txt += "\n\n### Algorithm\n"
txt += get_func_blurb(run_calibration)

KeyError: 'REL_SEV_UP'

In [ ]:
display(Markdown(txt))

In [ ]:
txt = "# Selection of countries for inclusion\n"
either_mob_avail, summary, g_avail, fb_avail = get_mob_avail_countries()
txt += get_func_blurb(get_mob_avail_countries)
death_data, case_data = gather_who_data(either_mob_avail)
txt += get_func_blurb(gather_who_data) + "\n\n"
no_deaths, no_cases = find_absent_inds(death_data, case_data, summary)
txt += get_func_blurb(find_absent_inds)
neg_deaths, neg_cases = find_neg_inds(death_data, case_data, summary)
txt += get_func_blurb(find_neg_inds)
death_outliers, case_outliers = find_outliers(death_data, case_data, summary)
death_nans, case_nans, death_repeats, case_repeats = find_nans_repeats(death_data, case_data, summary)
txt += get_func_blurb(find_nans_repeats) + "\n\n"
txt += "The final results of the selection are presented in @tbl-inc. "
txt += "The final included countries are illustrated in @fig-inc below. "
display(Markdown(txt))

In [ ]:
excluded = set(no_deaths + no_cases + neg_deaths + neg_cases + death_nans + case_nans + death_repeats + case_repeats + death_outliers + case_outliers)
included = [c for c in either_mob_avail if c not in excluded]
add_bool_row_to_table(summary, included, "Incl-uded")

In [ ]:
#| label: fig-inc
#| fig-cap: Included countries and mobility availability. Countries coloured according to mobility availability. Neither source available, grey; Google mobility only available, green; Facebook mobility only available, blue; both sources available, purple. Red markers indicate included in analysis.
world = get_world_shp()
world["geometry"] = world.simplify(tolerance=0.1)
add_mob_avail_to_world(world, g_avail, fb_avail)
world["included"] = world["ISO_A3"].isin(included)
plot_inclusion(world);

In [ ]:
#| label: tbl-inc
#| tbl-cap: Reasons for country inclusion and exclusion.
summary.index = summary.index.map(lambda iso3: pycountry.countries.lookup(iso3).name)
display(Markdown(summary.sort_index().to_markdown()))

\newpage
# Parameter choices and supporting evidence

In [ ]:
loaded_priors = yml.safe_load(open(BASE_PATH / "data/evidence/priors.yml", "r"))
duration_params = loaded_priors["durations"]

## Time period parameters

In [ ]:
col_widths = '{tbl-colwidths="[12, 7, 7, 74]"}'
durations_df = get_table_df_from_priors_dict(loaded_priors["durations"])
keep_cols = [c for c in durations_df if c != "Short_name"]
dur_table = durations_df[keep_cols]
caption = ": Parameters and supporting evidence to time period priors. "
Markdown(dur_table.to_markdown() + "\n" + caption + col_widths)

In [ ]:
#| fig-cap: Duration-related parameter prior distributions.
plot_duration_params(duration_params)

{{< pagebreak >}}

## Proportion parameters

In [ ]:
betas_df = get_table_df_from_priors_dict(loaded_priors["beta"])
caption = ": Parameters and supporting evidence to beta-distributed priors. "
Markdown(betas_df.to_markdown() + "\n" + caption + col_widths)

In [ ]:
#| fig-cap: Proportion parameter prior distributions. Note horizontal axis range differs between upper two panels and lower three panels.
plot_beta_priors(loaded_priors["beta"])

In [ ]:
fixed_params = loaded_priors["fixed"]
params_df = pd.DataFrame.from_dict(fixed_params).T
params_df = params_df.set_index("param_name")
params_df.columns = params_df.columns.str.capitalize()
params_df.index.name = None
caption = "\n: Fixed parameters and supporting evidence. "

{{< pagebreak >}}

## Fixed parameters

In [ ]:
Markdown(params_df.to_markdown() + caption + '{tbl-colwidths="[12, 7, 81]"}')

In [ ]:
loaded_priors["other"]["relseverity"]

{{< pagebreak >}}

## Between-variant relative parameters

In [ ]:
Markdown(loaded_priors["other"]["relinfect"]["evidence"])
Markdown(loaded_priors["other"]["relseverity"]["evidence"])

{{< pagebreak >}}

# References